# Data Preparation: UrbanSound8K

This notebook downloads and preveiws the dataset.  
According to the protocol, **we must not shuffle UrbanSound8K randomly**. The dataset provides 10 predefined folds. We will preserve those folds and later evaluate models with 10-folds-cross-validation.

In [ ]:
from pathlib import Path
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

ROOT = Path.cwd().parent.resolve()
DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Raw data dir:", RAW_DIR)
print("Processed data dir:", PROCESSED_DIR)

## Downloading the Dataset

In [ ]:
import soundata

dataset = soundata.initialize(
    "urbansound8k", data_home=str(RAW_DIR / "UrbanSound8K_soundata")
)

In [ ]:
dataset.download()

In [ ]:
dataset.validate()

## Loading Metadata

In [ ]:
DATASET_DIR = RAW_DIR / "UrbanSound8K_soundata"
METADATA_DIR = DATASET_DIR / "metadata/UrbanSound8K.csv"


def load_metadata(metadata_dir: Path) -> pd.DataFrame:
    df = pd.read_csv(metadata_dir)
    df["audio_path"] = df.apply(
        lambda row: str(
            DATASET_DIR / f"audio/fold{row['fold']}" / row["slice_file_name"]
        ),
        axis=1,
    )
    return df


metadata = load_metadata(METADATA_DIR)
print(metadata.shape)
display(metadata.head())

## Basic Integrity Check

In [ ]:
print("Number of clips:", len(metadata))
print("Number of classes:", len(metadata["class"].unique()))
print("Classes:", sorted(metadata["class"].unique()))
print("Folds:", sorted(int(x) for x in metadata["fold"].unique()))

missing_files = metadata[~metadata["audio_path"].apply(lambda p: Path(p).exists())]
print("Missing audio files:", len(missing_files))

missing_labels = metadata["class"].isna().sum()
print("Missing labels:", missing_labels)

metadata[["slice_file_name", "fold", "classID", "class", "audio_path"]].head()

## Class Distribution

In [ ]:
class_counts = metadata["class"].value_counts().sort_values(ascending=False)
display(class_counts.to_frame("count"))

plt.figure(figsize=(10, 5))
class_counts.plot(kind="bar")
plt.title("UrbanSound8K Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of clips")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Fold Distribution

In [ ]:
class_counts = metadata["fold"].value_counts().sort_values(ascending=False)
display(class_counts.to_frame("count"))

plt.figure(figsize=(8, 4))
class_counts.plot(kind="bar")
plt.title("UrbanSound8K Fold Distribution")
plt.xlabel("Fold")
plt.ylabel("Number of clips")
plt.tight_layout()
plt.show()

## Class Distribution per Fold

In [ ]:
fold_class_table = pd.crosstab(metadata["fold"], metadata["class"])
display(fold_class_table)

plt.figure(figsize=(12, 6))
plt.imshow(fold_class_table.values, aspect="auto")
plt.colorbar(label="Number of clips")
plt.title("Class Distribution per Fold")
plt.xlabel("Class")
plt.ylabel("Fold")
plt.xticks(
    range(len(fold_class_table.columns)),
    fold_class_table.columns,
    rotation=45,
    ha="right",
)
plt.yticks(range(len(fold_class_table.columns)), fold_class_table.index)
plt.tight_layout()
plt.show()

## Audio Duration and Sampling Rate Inspection

In [ ]:
import librosa


def inspect_audio_file(path):
    y, sr = librosa.load(path, sr=None, mono=False)
    if y.ndim == 1:
        n_samples = y.shape[0]
        n_channels = 1
    else:
        n_channels = y.shape[0]
        n_samples = y.shape[1]
    duration = n_samples / sr
    return duration, sr, n_channels


samples_df = metadata.sample(min(300, len(metadata)), random_state=10).copy()

durations = []
sample_rates = []
channels = []

for path in tqdm(samples_df["audio_path"], desc="Inspecting audio"):
    duration, sr, n_channels = inspect_audio_file(path)
    durations.append(duration)
    sample_rates.append(sr)
    channels.append(n_channels)

samples_df["duration"] = durations
samples_df["sample_rate"] = sample_rates
samples_df["n_channels"] = channels

display(
    samples_df[
        ["slice_file_name", "class", "fold", "duration", "sample_rate", "n_channels"]
    ].head()
)
display(samples_df[["duration", "sample_rate", "n_channels"]].describe())

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(samples_df["duration"], bins=30)
plt.title("Sampled Clip Duration Distribution")
plt.xlabel("Duration, seconds")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
samples_df["sample_rate"].value_counts().sort_index().plot(kind="bar")
plt.title("Sampled Sampling Rate Distribution")
plt.xlabel("Sampling rate")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## Creating 10-fold cross-validation Plan

In [ ]:
cv_plan = []

for test_fold in sorted(metadata["fold"].unique()):
    train_folds = [int(f) for f in sorted(metadata["fold"].unique()) if f != test_fold]
    cv_plan.append(
        {
            "test_fold": int(test_fold),
            "train_folds": train_folds,
            "n_train": int(metadata[metadata["fold"].isin(train_folds)].shape[0]),
            "n_test": int(metadata[metadata["fold"] == test_fold].shape[0]),
        }
    )

cv_plan_df = pd.DataFrame(cv_plan)
display(cv_plan_df)

cv_plan_dir = PROCESSED_DIR / "cv_plan_10fold.json"
with open(cv_plan_dir, "w") as f:
    json.dump(cv_plan, f, indent=2)

print("Saved:", cv_plan_dir)

## Saving Cleaned Metadata

In [ ]:
metadata_clean = metadata.copy()

preferred_cols = [
    "slice_file_name",
    "fsID",
    "start",
    "end",
    "salience",
    "fold",
    "classID",
    "class",
    "audio_path",
]
existing_cols = [c for c in preferred_cols if c in metadata_clean.columns]
other_cols = [c for c in metadata_clean.columns if c not in existing_cols]
metadata_clean = metadata_clean[existing_cols + other_cols]

metadata_dir = PROCESSED_DIR / "urbansound8k_metadata_clean.csv"
metadata_clean.to_csv(metadata_dir, index=False)

print("Saved", metadata_dir)
metadata_clean.head()

## Summary for the Report

In [ ]:
summary = {
    "n_clips": int(len(metadata_clean)),
    "n_classes": int(metadata_clean["class"].nunique()),
    "classes": sorted(metadata_clean["class"].unique().tolist()),
    "folds": sorted([int(x) for x in metadata_clean["fold"].unique()]),
    "evaluation_protocol": "10-fold cross-validation using the predefined UrbanSound8K folds. In each run, 9 folds are used for training and 1 fold for testing.",
}

summary_path = PROCESSED_DIR / "dataset_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("Saved:", summary_path)